# 🏢 Enterprise AP-Env: Training Notebook

**Meta AI Hackathon Grand Finale**

This notebook trains a rule-based agent on the Enterprise Accounts Payable environment across all 5 task difficulties and produces reward curves showing agent improvement.

## What this environment does
An AI agent is dropped into an enterprise AP department and must:
- Read emails from an inbox to find invoices
- Query an ERP system (with schema drift)
- Extract structured fields from unstructured invoice text
- Detect anomalies: price mismatches, tax fraud, duplicate invoices, phishing
- Negotiate with vendors via email
- Make a final approve/reject decision

## Tasks
| Task | Difficulty | Description |
|------|-----------|-------------|
| `easy` | Beginner | Read email → Query ERP → Extract fields → Approve |
| `medium` | Medium | Detect price mismatch → Reject |
| `hard` | Hard | Handle ERP schema drift + detect duplicate invoice |
| `expert_negotiation` | Expert | Email vendor, get corrected invoice, approve |
| `expert_fraud` | Expert | Detect lookalike domain phishing attack |

## Step 1: Install Dependencies

In [ ]:
!pip install fastapi uvicorn pydantic openai python-dotenv requests gradio matplotlib --quiet
print('✅ Dependencies installed')

## Step 2: Clone the Repository

In [ ]:
import os

if not os.path.exists('invoice-env'):
    !git clone https://github.com/dharmendra26-wiz/invoice-env
    print('✅ Repo cloned')
else:
    !cd invoice-env && git pull
    print('✅ Repo updated')

os.chdir('invoice-env')
print(f'📁 Working directory: {os.getcwd()}')

## Step 3: Verify the Environment Works

In [ ]:
import sys
sys.path.insert(0, '.')

from app.environment import InvoiceEnvironment
from app.models import Action

# Quick smoke test
for task in ['easy', 'medium', 'hard', 'expert_negotiation', 'expert_fraud']:
    env = InvoiceEnvironment(task)
    obs = env.reset()
    print(f'✅ {task:25s} | inbox={len(obs.inbox_status)} emails | step={obs.current_step}')

print('\n🎉 All 5 tasks initialized successfully!')

## Step 4: Run Training (60 Episodes per Task)

In [ ]:
!python train.py --episodes 60

## Step 5: Display Reward Curves

In [ ]:
from IPython.display import Image, display
display(Image('reward_curves.png', width=900))

## Step 6: Show Final Scores

In [ ]:
import json

with open('training_results.json') as f:
    results = json.load(f)

print('=' * 50)
print('  FINAL TRAINING RESULTS')
print('=' * 50)
targets = {'easy': 0.85, 'medium': 0.75, 'hard': 0.65, 'expert_negotiation': 0.70, 'expert_fraud': 0.70}
all_pass = True
for task, score in results['final_scores'].items():
    target = targets[task]
    status = 'PASS' if score >= target else 'FAIL'
    if score < target:
        all_pass = False
    print(f'  [{status}] {task:<25} score={score:.3f}  target={target}')

print('=' * 50)
if all_pass:
    print('  🎉 ALL TASKS PASSED!')
else:
    print('  ⚠️  Some tasks below target - try more episodes')

## Step 7: Run a Single Episode Manually
Watch the agent take actions step by step on the `expert_fraud` task:

In [ ]:
import re, sys
sys.path.insert(0, '.')
from app.environment import InvoiceEnvironment
from app.models import Action

def parse_vendor(body):
    m = re.search(r'Vendor:\s*(.+)', body, re.IGNORECASE)
    return m.group(1).strip() if m else 'Unknown'

task_name = 'expert_fraud'
env = InvoiceEnvironment(task_name)
obs = env.reset()
print(f'\n--- Running {task_name} episode ---')
print(f'Inbox: {[e["sender"] for e in obs.inbox_status]}')
print(f'\nNOTICE: Check if the sender domain looks suspicious!')

# Step 1: Read email
r = env.step(Action(action_type='read_email', email_id=obs.inbox_status[0]['id']))
print(f'\nStep 1 - READ EMAIL: reward={r.reward}')

# Step 2: Query ERP
vendor = parse_vendor(r.observation.email_content or '')
r = env.step(Action(action_type='query_erp', api_endpoint='/api/v1/po', api_payload={'vendor_name': vendor}))
print(f'Step 2 - QUERY ERP ({vendor}): reward={r.reward}')

# Step 3: Flag as fraud
r = env.step(Action(action_type='flag', field_name='fraud'))
print(f'Step 3 - FLAG FRAUD: reward={r.reward}')

# Step 4: Reject
r = env.step(Action(action_type='reject'))
print(f'Step 4 - REJECT: final_score={r.reward}')
print(f'\nResult: {r.observation.message}')

## Summary

This environment trains agents to handle real-world enterprise workflows:
- **Multi-app navigation**: Email + ERP + Vendor communication
- **Schema drift adaptation**: ERP API changes its contract mid-workflow  
- **Multi-agent negotiation**: Agent emails vendor, vendor replies with corrected invoice
- **Security/fraud detection**: Lookalike phishing domain detection

Built for the **Meta AI Hackathon Grand Finale 2026** 🏆